# 01 - Prepare ClinVar Binary Variant Dataset

Purpose: download and preprocess ClinVar GRCh38 VCF data for binary variant classification.

This notebook creates a research-only dataset. It is not a medical diagnosis workflow and must not be used for clinical decision-making.

## 1. Install Dependencies

The notebook uses standard Python data tooling plus lightweight genomics helpers. The VCF parsing below is still done manually with `gzip` through `training/utils/clinvar_parser.py`.

In [ ]:
!pip install -q pandas numpy scikit-learn biopython requests tqdm pyfaidx datasets

## 2. Notebook Setup

Run this notebook from the repository root in Colab. If you cloned the repo into `/content/variant-risk-explainer`, the setup cell will find it automatically.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# Imported to verify the requested Colab dependencies are available.
import Bio
from datasets import Dataset
from pyfaidx import Fasta

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "training").exists():
    colab_candidate = Path("/content/variant-risk-explainer")
    if colab_candidate.exists():
        PROJECT_ROOT = colab_candidate

helper_path = PROJECT_ROOT / "training" / "utils" / "clinvar_parser.py"
if not helper_path.exists():
    raise FileNotFoundError(
        "Could not find training/utils/clinvar_parser.py. Clone or upload this repository before running the notebook."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DIR}")
print(f"Processed data dir: {PROCESSED_DIR}")

## 3. Download ClinVar GRCh38 VCF

Source: `https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz`

In [ ]:
CLINVAR_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz"
VCF_PATH = RAW_DIR / "clinvar_grch38.vcf.gz"


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Using existing file: {destination}")
        return

    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))

    with destination.open("wb") as output, tqdm(
        total=total,
        unit="B",
        unit_scale=True,
        desc="Downloading ClinVar GRCh38 VCF",
    ) as progress:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if not chunk:
                continue
            output.write(chunk)
            progress.update(len(chunk))


download_file(CLINVAR_URL, VCF_PATH)
print(f"Downloaded VCF: {VCF_PATH}")
print(f"Size: {VCF_PATH.stat().st_size / (1024 * 1024):.2f} MB")

## 4. Parse VCF Manually With gzip

The helper parser opens the compressed VCF with `gzip`, skips `##` metadata lines and the final `#CHROM` header row, extracts `CHROM`, `POS`, `ID`, `REF`, `ALT`, and `INFO`, then parses these INFO fields:

- `CLNSIG`
- `GENEINFO`
- `CLNHGVS`, when available
- `CLNVC`, when available

In [ ]:
from training.utils.clinvar_parser import parse_clinvar_vcf

# Set MAX_RECORDS to a small integer, such as 50000, for a quick Colab smoke test.
# Use None for the full ClinVar VCF.
MAX_RECORDS = None

raw_df = parse_clinvar_vcf(VCF_PATH, max_records=MAX_RECORDS)
print(f"Parsed rows: {len(raw_df):,}")
display(raw_df.head())

## 5. Filter Variants and Create Binary Labels

Filtering rules:

- Keep SNVs where `len(REF) == 1` and `len(ALT) == 1`.
- Keep small indels where `abs(len(REF) - len(ALT)) <= 50`.
- Remove symbolic ALT values such as `<DEL>` and `<DUP>`.
- Remove variants with multiple ALT alleles for the MVP.
- Remove uncertain, conflicting, unsupported, and mixed labels.

Binary label rules:

- `Pathogenic` or `Likely_pathogenic` -> `label = 1`.
- `Benign` or `Likely_benign` -> `label = 0`.
- Drop rows containing `Conflicting_interpretations`, `Uncertain_significance`, `risk_factor`, `association`, `drug_response`, `protective`, or `not_provided`.

In [ ]:
from training.utils.clinvar_parser import prepare_binary_variants

clinvar_df = prepare_binary_variants(raw_df)

print(f"Rows after filtering and labeling: {len(clinvar_df):,}")
display(clinvar_df["label_name"].value_counts().rename_axis("label_name").reset_index(name="count"))
display(clinvar_df.head())

## 6. Add `variant_id`

The parser creates stable GRCh38 IDs in this format:

`GRCh38-{chrom}-{pos}-{ref}-{alt}`

In [ ]:
assert clinvar_df["variant_id"].str.startswith("GRCh38-").all()
duplicate_variant_ids = int(clinvar_df["variant_id"].duplicated().sum())
print(f"Duplicate variant_id rows: {duplicate_variant_ids:,}")

display(clinvar_df[["variant_id", "CHROM", "POS", "REF", "ALT", "label", "label_name"]].head())

## 7. Train, Validation, and Test Split

Split the dataset into 70 percent train, 15 percent validation, and 15 percent test sets. The split is stratified by the binary label and uses `random_state=42`.

In [ ]:
if clinvar_df["label"].nunique() != 2:
    raise ValueError("Expected both binary classes before stratified splitting.")

train_df, temp_df = train_test_split(
    clinvar_df,
    test_size=0.30,
    stratify=clinvar_df["label"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42,
)

print(f"Train rows: {len(train_df):,} ({len(train_df) / len(clinvar_df):.1%})")
print(f"Val rows:   {len(val_df):,} ({len(val_df) / len(clinvar_df):.1%})")
print(f"Test rows:  {len(test_df):,} ({len(test_df) / len(clinvar_df):.1%})")

## 8. Save Processed CSV Files

The files are saved under `data/processed/`:

- `clinvar_binary_variants.csv`
- `train.csv`
- `val.csv`
- `test.csv`

In [ ]:
processed_path = PROCESSED_DIR / "clinvar_binary_variants.csv"
train_path = PROCESSED_DIR / "train.csv"
val_path = PROCESSED_DIR / "val.csv"
test_path = PROCESSED_DIR / "test.csv"

clinvar_df.to_csv(processed_path, index=False)
train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

for path in [processed_path, train_path, val_path, test_path]:
    print(path.relative_to(PROJECT_ROOT))

## 9. Sanity Checks

Check class balance, SNV and indel counts, missing gene info, and example rows before using these files for training.

In [ ]:
def class_distribution(name: str, frame: pd.DataFrame) -> pd.DataFrame:
    counts = frame["label_name"].value_counts().rename_axis("label_name").reset_index(name="count")
    counts.insert(0, "split", name)
    counts["fraction"] = counts["count"] / len(frame)
    return counts


distribution_df = pd.concat(
    [
        class_distribution("all", clinvar_df),
        class_distribution("train", train_df),
        class_distribution("val", val_df),
        class_distribution("test", test_df),
    ],
    ignore_index=True,
)

snv_count = int((clinvar_df["variant_type"] == "SNV").sum())
indel_count = int((clinvar_df["variant_type"] == "INDEL").sum())
missing_gene_info_count = int(clinvar_df["gene_symbol"].isna().sum())

print("Class distribution")
display(distribution_df)

print(f"Number of SNVs: {snv_count:,}")
print(f"Number of indels: {indel_count:,}")
print(f"Missing gene info count: {missing_gene_info_count:,}")

print("Example rows")
display(
    clinvar_df[
        ["variant_id", "CHROM", "POS", "REF", "ALT", "variant_type", "gene_symbol", "CLNSIG", "label"]
    ].sample(min(10, len(clinvar_df)), random_state=42)
)

## 10. Sequence Extraction: +/-512 bp Flanking DNA

This section adds a `sequence` column to the train, validation, and test splits.

Two fetching modes are supported:

- **Mode A: UCSC API mode** uses `https://api.genome.ucsc.edu/getData/sequence` with `genome=hg38`.
- **Mode B: local FASTA mode** uses `pyfaidx` if you provide a local GRCh38 FASTA path.

For full ClinVar, the UCSC API can be slow. Start with `SAMPLE_MODE = True`, which processes 1000 variants across the existing splits first.

In [ ]:
from training.utils.sequence_fetcher import add_sequences_to_dataframe, build_sequence_fetcher

FLANK_SIZE = 512
MIN_SEQ_LEN = 200

# Keep this True for the first Colab run. Set to False for the full train/val/test splits.
SAMPLE_MODE = True
SAMPLE_SIZE = 1000

# Choose "ucsc" or "fasta".
SEQUENCE_FETCH_MODE = "ucsc"

# Required only when SEQUENCE_FETCH_MODE = "fasta".
LOCAL_FASTA_PATH = ""

# UCSC API settings. Increase sleep if you see rate limits or transient failures.
UCSC_SLEEP_SECONDS = 0.15
UCSC_MAX_RETRIES = 3
UCSC_CACHE_PATH = PROCESSED_DIR / f"ucsc_sequence_cache_flank{FLANK_SIZE}.json"

print(f"FLANK_SIZE: {FLANK_SIZE}")
print(f"MIN_SEQ_LEN: {MIN_SEQ_LEN}")
print(f"SAMPLE_MODE: {SAMPLE_MODE}")
print(f"SEQUENCE_FETCH_MODE: {SEQUENCE_FETCH_MODE}")
print(f"UCSC cache path: {UCSC_CACHE_PATH.relative_to(PROJECT_ROOT)}")

### 10.1 Build Split Inputs for Sequence Fetching

When `SAMPLE_MODE` is enabled, this cell samples up to 1000 variants across train, validation, and test while preserving split membership. Full mode uses every row.

In [ ]:
def combine_splits_for_sequence_fetching(
    train_frame: pd.DataFrame,
    val_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
) -> pd.DataFrame:
    frames = []
    for split_name, frame in [("train", train_frame), ("val", val_frame), ("test", test_frame)]:
        split_frame = frame.copy()
        split_frame["_split"] = split_name
        frames.append(split_frame)
    return pd.concat(frames, ignore_index=True)


combined_for_sequences = combine_splits_for_sequence_fetching(train_df, val_df, test_df)

if SAMPLE_MODE:
    sample_n = min(SAMPLE_SIZE, len(combined_for_sequences))
    stratify_values = combined_for_sequences["_split"].astype(str) + "_" + combined_for_sequences["label"].astype(str)

    if sample_n < len(combined_for_sequences) and stratify_values.value_counts().min() >= 2:
        _, sequence_source_df = train_test_split(
            combined_for_sequences,
            test_size=sample_n,
            stratify=stratify_values,
            random_state=42,
        )
    else:
        sequence_source_df = combined_for_sequences.sample(n=sample_n, random_state=42)
else:
    sequence_source_df = combined_for_sequences.copy()

sequence_source_splits = {}
for split_name in ["train", "val", "test"]:
    split_frame = sequence_source_df.loc[sequence_source_df["_split"] == split_name].copy()
    sequence_source_splits[split_name] = split_frame.drop(columns=["_split"])
    print(f"{split_name}: {len(sequence_source_splits[split_name]):,} variants selected")

### 10.2 Fetch Sequences

UCSC mode requests intervals with these parameters:

- `genome=hg38`
- `chrom=chrN`, with ClinVar chromosomes such as `1` converted to `chr1` and mitochondrial chromosomes converted to `chrM`
- `start=POS - 1 - FLANK_SIZE`
- `end=POS - 1 + len(REF) + FLANK_SIZE`

The fetcher uppercases each sequence, keeps only `A/C/G/T/N`, retries transient failures, sleeps between UCSC requests, and caches fetched sequence windows.

In [ ]:
if SEQUENCE_FETCH_MODE == "fasta" and not LOCAL_FASTA_PATH:
    raise ValueError("Set LOCAL_FASTA_PATH before using local FASTA mode.")

fetcher = build_sequence_fetcher(
    mode=SEQUENCE_FETCH_MODE,
    fasta_path=LOCAL_FASTA_PATH or None,
    cache_path=UCSC_CACHE_PATH if SEQUENCE_FETCH_MODE == "ucsc" else None,
    sleep_seconds=UCSC_SLEEP_SECONDS,
    max_retries=UCSC_MAX_RETRIES,
)

sequence_outputs = {}
sequence_stats = []

for split_name, split_frame in sequence_source_splits.items():
    with_sequences, stats = add_sequences_to_dataframe(
        split_frame,
        fetcher=fetcher,
        flank_size=FLANK_SIZE,
        min_seq_len=MIN_SEQ_LEN,
        progress_desc=f"{split_name}: fetching sequences",
    )
    sequence_outputs[split_name] = with_sequences
    stats["split"] = split_name
    sequence_stats.append(stats)

if hasattr(fetcher, "save_cache"):
    fetcher.save_cache()

sequence_stats_df = pd.DataFrame(sequence_stats)
display(sequence_stats_df[["split", "input_rows", "output_rows", "failed_fetches", "too_short", "dropped_rows"]])

### 10.3 Save Sequence-Enriched Splits

The final files are saved under `data/processed/`:

- `train_with_sequences.csv`
- `val_with_sequences.csv`
- `test_with_sequences.csv`

In [ ]:
train_with_sequences_path = PROCESSED_DIR / "train_with_sequences.csv"
val_with_sequences_path = PROCESSED_DIR / "val_with_sequences.csv"
test_with_sequences_path = PROCESSED_DIR / "test_with_sequences.csv"

sequence_outputs["train"].to_csv(train_with_sequences_path, index=False)
sequence_outputs["val"].to_csv(val_with_sequences_path, index=False)
sequence_outputs["test"].to_csv(test_with_sequences_path, index=False)

for path in [train_with_sequences_path, val_with_sequences_path, test_with_sequences_path]:
    print(path.relative_to(PROJECT_ROOT))

### 10.4 Sequence Quality Checks

Review sequence length distribution, failed fetches, class distribution after sequence filtering, and the first five examples.

In [ ]:
sequence_quality_frames = []
for split_name, split_frame in sequence_outputs.items():
    if split_frame.empty:
        continue
    quality_frame = split_frame.copy()
    quality_frame["split"] = split_name
    quality_frame["sequence_length"] = quality_frame["sequence"].str.len()
    sequence_quality_frames.append(quality_frame)

if sequence_quality_frames:
    sequence_quality_df = pd.concat(sequence_quality_frames, ignore_index=True)

    print("Sequence length distribution")
    display(sequence_quality_df.groupby("split")["sequence_length"].describe())

    print("Number of failed fetches")
    display(sequence_stats_df[["split", "failed_fetches", "too_short", "dropped_rows"]])

    print("Class distribution after sequence filtering")
    display(
        sequence_quality_df.groupby(["split", "label_name"])
        .size()
        .rename("count")
        .reset_index()
    )

    print("First 5 examples")
    display(
        sequence_quality_df[
            ["split", "variant_id", "CHROM", "POS", "REF", "ALT", "label", "label_name", "sequence"]
        ].head(5)
    )
else:
    print("No sequences were fetched. Check API connectivity, FASTA path, or input intervals.")

## Next Step

Use `data/processed/train_with_sequences.csv`, `data/processed/val_with_sequences.csv`, and `data/processed/test_with_sequences.csv` as sequence-ready inputs for DNABERT-2 fine-tuning. Keep GRCh38 coordinates and reference assets consistent throughout the pipeline.